In [ ]:
%run ./dataset.ipynb

In [ ]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torchvision.transforms import v2

In [ ]:
def train_transform(input_size):
    """Augmentation for training. Ends with a float tensor in 0-255 (no /255): the shipped models and the service's
    inference use 0-255, so v2.ToDtype(scale=False) instead of ToTensor."""
    return v2.Compose([
        v2.ToPILImage(),
        v2.RandomResizedCrop(size=tuple(input_size), scale=(0.9, 1.1)),
        v2.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
        v2.RandomRotation(10),
        v2.RandomHorizontalFlip(),
        v2.PILToTensor(),
        v2.ToDtype(torch.float32, scale=False),
    ])

In [ ]:
def get_train_loader(conf):
    root_path = Path(conf.train_root_path) / conf.patch_info
    trainset = DatasetFolderFT(root_path, train_transform(conf.input_size),
                               None, conf.ft_width, conf.ft_height)
    train_loader = DataLoader(
        trainset,
        batch_size=conf.batch_size,
        shuffle=True,
        pin_memory=True,
        num_workers=conf.num_workers)
    return train_loader